In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "random_20pages.pdf"
pages = PyPDFLoader(PDF_PATH).load()

print(f"Loaded {len(pages)} pages from pdf")
print(pages[0].page_content[:100])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
)
chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

In [ ]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_chroma import Chroma

class LocalEmbeddings:
    def __init__(self):
        self._ef = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [list(v) for v in self._ef(texts)]
    def embed_query(self, text):
        return list(self._ef([text])[0])

embeddings = LocalEmbeddings()
vector_store = Chroma.from_documents(chunks, embeddings)
print("Vector store ready")

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "what is the role of maths in deep ocean history?"
retrieved_with_scores = vector_store.similarity_search_with_relevance_scores(test_query, k=3)

for i, (doc, score) in enumerate(retrieved_with_scores):
    print(f"... chunk {i} (Score: {score:.4f}) ..")
    print(doc.page_content[:300])
    print(".....")

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")

class PersonInfo(BaseModel):
    name: str | None = Field(description="The name of the person.")
    age: str | None = Field(description="The age of the person.")

pydantic_parser = PydanticOutputParser(pydantic_object=PersonInfo)

SYS_PROMPT = """
Answer using the provided context only.
{format_instructions}

Context:
{ctxt}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    ("human", "{question}")
]).partial(format_instructions=pydantic_parser.get_format_instructions())

structured_chain = prompt | llm | pydantic_parser

def ask_question(question_txt):
    docs = retriever.invoke(question_txt)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    return structured_chain.invoke({"ctxt": context, "question": question_txt})

print("Ready")

In [ ]:
import json

res = ask_question("who is mentioned as a key figure in the origins of the universe?")

print(json.dumps(res.model_dump(), indent=2))